In [1]:
import os
import sys
from pathlib import Path


nb_dir = Path.cwd()
        

target = (nb_dir / '..' / '..').resolve()
os.chdir(target)




In [2]:
from grasp.graph.graph_storage import GraphStorage
from grasp.graph.graph_storage import load_graph_storage

gs_path = "data/optc_051/graph_storage/optc_051_default_experiment_dataset-optc_051_context_size-120_step_size-120_graph_storage.pt"

gs: GraphStorage = load_graph_storage(gs_path)


print(gs)
known_executables = gs.train_subject_cmds
print(len(known_executables))
print(len(known_executables))

64980
64980


In [3]:
gs.train_subject_cmds

['/Device/HarddiskVolume1/Windows/Explorer.EXE',
 '/Device/HarddiskVolume1/Windows/system32/SearchFilterHost.exe',
 'Idle',
 'System',
 'System',
 'MemCompression',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 'taskhostw.exe',
 'taskhostw.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidpolicyconverter.exe',
 '/Device/HarddiskVolume1/Windows/system32/lsass.exe',
 '/Device/HarddiskVolume1/lwabeat//lwabeat.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Windows/System32/svchost.exe',
 '/Device/HarddiskVolume1/Windows/system32/svchost.exe',
 '/Device/HarddiskVolume1/Program Files/Windows Defender/MsMpEng.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidcertstorecheck.exe',
 '/Device/HarddiskVolume1/Windows/system32/appidcertstorecheck.exe',
 '/Device/HarddiskVolume1/Program Files/Windows Defender/MsMpEng.exe',
 '/Device/HarddiskVolume1/Windows/system32/conhost.exe',
 '/Device/HarddiskVolume1/Windows/system32/RAServer.exe',
 '

In [4]:
gs.train_subject_cmd_to_id

{'%SystemRoot%/system32/csrss.exe': 0,
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe': 1,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe': 2,
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe': 3,
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe': 4,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler.exe': 5,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe': 6,
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe': 7,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE': 8,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE': 9,
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE': 10,
 '/Device/HarddiskVolume1/Program Files/Microsoft Office/Office15/msoia.exe': 11,
 '/D

In [5]:
len(set(known_executables))

148

In [6]:
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)
print(len(known_executables_list))
known_executables_list


148


['%SystemRoot%/system32/csrss.exe',
 '//?/C:/Program Files (x86)/Mozilla Firefox/firefox.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/AcroRd32.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Adobe/Reader 9.0/Reader/reader_sl.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Common Files/Adobe/ARM/1.0/AdobeARM.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/1.3.26.9/GoogleCrashHandler64.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Google/Update/GoogleUpdate.exe',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/EXCEL.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/POWERPNT.EXE',
 '/Device/HarddiskVolume1/Program Files (x86)/Microsoft Office/Office15/WINWORD.EXE',
 '/Device/HarddiskVolume1/Program Files/Microsoft Office/Office15/msoia.exe',
 '/Device/HarddiskVolume1/Program Files/VM

In [7]:
import time
from urllib.parse import urlparse, unquote

import psycopg2
from psycopg2 import sql

from grasp import config
from grasp.schema import DatasetName

user = "postgres"
password = "lolroflomg"
host = config.DB_HOST
port = 9889

base_url = f"postgresql://{user}:{password}@{host}:{port}"
connection_real_data = f"{base_url}/{DatasetName.OPTC_051.value}"

# Known executable commands from training graph storage
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)

new_table_name = "subject_node_table"
backup_table_name = f"{new_table_name}_backup"

# Parse connection URL once
parsed = urlparse(connection_real_data)
dbname = parsed.path.lstrip('/') if parsed.path else None
db_user = unquote(parsed.username) if parsed.username else None
db_password = unquote(parsed.password) if parsed.password else None
db_host = parsed.hostname
db_port = parsed.port

t0 = time.perf_counter()
conn = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_process_unknown_exec_fast",
)

try:
    with conn:
        with conn.cursor() as cur:
            # Fast path: keep one immutable backup, recreate filtered table from it.
            # If backup already exists, reuse it. If not, rename original once.
            cur.execute("SELECT to_regclass(%s)", (new_table_name,))
            has_main = cur.fetchone()[0] is not None
            cur.execute("SELECT to_regclass(%s)", (backup_table_name,))
            has_backup = cur.fetchone()[0] is not None

            if not has_main and not has_backup:
                raise RuntimeError(
                    f"Neither '{new_table_name}' nor '{backup_table_name}' exists."
                )

            if has_main and not has_backup:
                t_rename = time.perf_counter()
                cur.execute(
                    sql.SQL("ALTER TABLE {} RENAME TO {}").format(
                        sql.Identifier(new_table_name),
                        sql.Identifier(backup_table_name),
                    )
                )
                print(
                    f"Renamed original table to backup in "
                    f"{time.perf_counter() - t_rename:.3f}s"
                )
            elif has_main and has_backup:
                # Keep existing backup as source of truth; refresh working table below.
                print(
                    f"Both '{new_table_name}' and '{backup_table_name}' exist; "
                    "keeping backup and refreshing working table."
                )

            source_table = backup_table_name if has_backup or has_main else new_table_name

            # Remember node_uuids that will be removed (unknown execs + NULL cmd)
            t_collect = time.perf_counter()
            cur.execute(
                sql.SQL(
                    """
                    SELECT node_uuid
                    FROM {}
                    WHERE path IS NULL OR NOT (path = ANY(%s))
                    """
                ).format(sql.Identifier(source_table)),
                (known_executables_list,),
            )
            deleted_node_hash_ids = [row[0] for row in cur.fetchall()]
            print(
                f"Collected {len(deleted_node_hash_ids)} deleted node_hash_ids in "
                f"{time.perf_counter() - t_collect:.3f}s"
            )

            t_rebuild = time.perf_counter()
            cur.execute(
                sql.SQL("DROP TABLE IF EXISTS {}").format(
                    sql.Identifier(new_table_name)
                )
            )
            cur.execute(
                sql.SQL("CREATE TABLE {} (LIKE {} INCLUDING ALL)").format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                )
            )
            cur.execute(
                sql.SQL(
                    "INSERT INTO {} SELECT * FROM {} WHERE path = ANY(%s)"
                ).format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(source_table),
                ),
                (known_executables_list,),
            )

            inserted_rows = cur.rowcount
            print(
                f"Rebuilt filtered '{new_table_name}' with {inserted_rows} rows in "
                f"{time.perf_counter() - t_rebuild:.3f}s"
            )

            # Useful sanity numbers
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(
                    sql.Identifier(source_table)
                )
            )
            source_count = cur.fetchone()[0]
            print(f"Source rows: {source_count}")
            print(f"Deleted rows: {source_count - inserted_rows}")

finally:
    conn.close()

print(f"Total elapsed: {time.perf_counter() - t0:.3f}s")

Renamed original table to backup in 0.000s
Collected 139 deleted node_hash_ids in 0.009s
Rebuilt filtered 'subject_node_table' with 50487 rows in 0.221s
Source rows: 50626
Deleted rows: 139
Total elapsed: 0.256s


In [8]:
deleted_node_hash_ids

['3d063387-b4b8-4163-bc64-5545fb5fecde',
 '52562a2b-2f90-44ca-b7ee-0ada5290a844',
 '36c5238e-a2c2-42cf-b5d1-56488cdf6c82',
 '6b1fe84b-d505-4748-8c51-dbbdce209602',
 '1fe1e8e4-7769-4d5a-a5a4-cd943538f2ad',
 'cf6c7753-d9da-4ebd-a38c-cbb8d1b6d54e',
 '04dbf3ad-aafc-4968-902f-73d0f9761074',
 'a321083f-0a4e-479e-8578-0e26cde1acea',
 '7398dbee-0f12-4ffe-8e59-5b397ec57029',
 '865c88c3-fc23-4a5e-81f7-3106a57bd433',
 '89d8a0b2-5b72-402a-b50a-66f596583f53',
 '4986cc39-9cde-46cf-a6b1-d851afe7b42f',
 'a087521e-0b24-467d-9e70-a724df8bd1ed',
 '9cd219ed-ecc9-4426-ad24-4fbdb1f4fd3b',
 '14f154fe-e587-44e4-9402-55b58681885d',
 '1d43b16c-9c16-4355-a23d-856ccfc31adf',
 'ffea2b80-71eb-46cf-8e9c-c4f868d662f1',
 '0a10b5f4-de5a-4ab8-aac9-4954c811e6d4',
 'cc5f647d-9b84-4851-abf1-604ebf727db4',
 '3d0a9313-d40d-41d4-86cc-4aa2ad8e5e9b',
 '2ff2b40a-7916-4c1a-8f77-421dac210de3',
 '6a5df088-e752-42ba-98e7-0e5185387581',
 'a04800b1-eedb-4187-a910-6aa9c88a64c7',
 'aedd6298-233c-4d22-98c8-8367e09e82f6',
 'edafc97f-93c1-

In [9]:
event_table_name = "event_table"

# Deduplicate once for stable/efficient ANY() checks
deleted_node_hash_ids_list = sorted(set(deleted_node_hash_ids))

# Process in batches to reduce memory pressure
BATCH_SIZE = 10000
deleted_batches = [
    deleted_node_hash_ids_list[i : i + BATCH_SIZE]
    for i in range(0, len(deleted_node_hash_ids_list), BATCH_SIZE)
]

t0_event = time.perf_counter()
conn_event = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_events_with_deleted_nodes_main_only",
)

try:
    with conn_event:
        with conn_event.cursor() as cur_event:
            # Ensure main table exists
            cur_event.execute("SELECT to_regclass(%s)", (event_table_name,))
            has_event_main = cur_event.fetchone()[0] is not None
            if not has_event_main:
                raise RuntimeError(f"Table '{event_table_name}' does not exist.")

            # Count source rows
            cur_event.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(event_table_name))
            )
            event_source_count = cur_event.fetchone()[0]

            # Count and delete rows in batches
            t_event_count = time.perf_counter()
            total_event_rows_to_delete = 0
            total_event_deleted_rows = 0

            for batch_idx, batch in enumerate(deleted_batches):
                try:
                    cur_event.execute(
                        sql.SQL(
                            """
                            SELECT COUNT(*)
                            FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    batch_rows_to_delete = cur_event.fetchone()[0]
                    total_event_rows_to_delete += batch_rows_to_delete

                    cur_event.execute(
                        sql.SQL(
                            """
                            DELETE FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                        ).format(sql.Identifier(event_table_name)),
                        (batch, batch),
                    )
                    total_event_deleted_rows += cur_event.rowcount
                    conn_event.commit()

                    print(
                        f"Batch {batch_idx + 1}/{len(deleted_batches)}: "
                        f"Deleted {cur_event.rowcount} rows "
                        f"(counted {batch_rows_to_delete} to delete) "
                        f"in {time.perf_counter() - t_event_count:.3f}s"
                    )

                except psycopg2.errors.DiskFull as e:
                    print(f"Disk full error in batch {batch_idx}. Stopping gracefully.")
                    conn_event.rollback()
                    raise

            print(
                f"Rows to delete from '{event_table_name}': {total_event_rows_to_delete} "
                f"(counted in {time.perf_counter() - t_event_count:.3f}s)"
            )
            print(f"Deleted {total_event_deleted_rows} rows from '{event_table_name}'")
            print(f"Event source rows: {event_source_count}")
            print(f"Event remaining rows: {event_source_count - total_event_deleted_rows}")

finally:
    conn_event.close()

print(f"Total event-table elapsed: {time.perf_counter() - t0_event:.3f}s")


Batch 1/1: Deleted 17582 rows (counted 17582 to delete) in 6.265s
Rows to delete from 'event_table': 17582 (counted in 6.265s)
Deleted 17582 rows from 'event_table'
Event source rows: 31611029
Event remaining rows: 31593447
Total event-table elapsed: 7.086s
